# 🏥 Healthcare Patient Analytics using PySpark

## End-to-End Google Colab Project

**Business Flow**

Patient Records → Large Dataset → PySpark → Data Cleaning → Patient Analysis → Medical Condition Analysis → Treatment Cost Analysis → Patient Segmentation → Readmission Analysis → Business Insights

> This notebook uses **synthetic healthcare data** for educational purposes. No real patient/medical data is used.

In [6]:
!pip install -q pyspark

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import random
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

spark = SparkSession.builder \
    .appName("HealthcarePatientAnalytics") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print("PySpark started successfully")
print("Spark version:", spark.version)

PySpark started successfully
Spark version: 4.0.4


## 1. Create a Large Synthetic Healthcare Dataset

We generate **1 million patient visit records** representing hospital visits.

Each patient can have multiple visits.

In [8]:
random.seed(42)

NUM_RECORDS = 1_000_000
NUM_PATIENTS = 200_000

genders = ["Male", "Female", "Other"]

blood_groups = ["A+", "A-", "B+", "B-", "AB+", "AB-", "O+", "O-"]

conditions = [
    "Diabetes",
    "Hypertension",
    "Heart Disease",
    "Asthma",
    "Cancer",
    "Kidney Disease",
    "Arthritis",
    "Migraine"
]

hospitals = [
    "City Hospital",
    "Apollo Care",
    "Metro Hospital",
    "Green Valley Hospital",
    "Sunrise Medical Center"
]

departments = [
    "Cardiology",
    "General Medicine",
    "Neurology",
    "Orthopedics",
    "Oncology",
    "Pulmonology",
    "Nephrology"
]

medications = [
    "Metformin",
    "Amlodipine",
    "Atorvastatin",
    "Salbutamol",
    "Paracetamol",
    "Insulin",
    "Aspirin",
    "Omeprazole"
]

insurance_types = [
    "Private",
    "Government",
    "Self-Pay",
    "Corporate"
]

cities = [
    "Chennai",
    "Bangalore",
    "Mumbai",
    "Delhi",
    "Hyderabad",
    "Pune",
    "Coimbatore"
]

start_date = datetime(2025, 1, 1)

data = []

for i in range(NUM_RECORDS):

    patient_id = random.randint(1, NUM_PATIENTS)

    age = random.randint(1, 90)

    gender = random.choice(genders)
    blood_group = random.choice(blood_groups)
    condition = random.choice(conditions)

    admission_date = start_date + timedelta(
        days=random.randint(0, 364)
    )

    hospital = random.choice(hospitals)
    department = random.choice(departments)

    length_of_stay = random.randint(1, 20)

    treatment_cost = __builtins__.round(
        random.uniform(5000, 150000) +
        (length_of_stay * random.uniform(500, 3000)),
        2
    )

    medication = random.choice(medications)
    insurance = random.choice(insurance_types)

    # Synthetic readmission logic:
    # longer stay and older age slightly increase probability
    readmission_probability = 0.08

    if age >= 60:
        readmission_probability += 0.08

    if length_of_stay >= 10:
        readmission_probability += 0.08

    if condition in ["Heart Disease", "Cancer", "Kidney Disease"]:
        readmission_probability += 0.08

    readmitted = (
        "Yes"
        if random.random() < readmission_probability
        else "No"
    )

    city = random.choice(cities)

    data.append((
        i + 1,
        patient_id,
        age,
        gender,
        blood_group,
        condition,
        admission_date.strftime("%Y-%m-%d"),
        hospital,
        department,
        treatment_cost,
        length_of_stay,
        medication,
        insurance,
        readmitted,
        city
    ))

print(f"Generated {len(data):,} synthetic healthcare records")

Generated 1,000,000 synthetic healthcare records


## 2. Create the PySpark DataFrame

In [9]:
schema = StructType([
    StructField("visit_id", IntegerType(), False),
    StructField("patient_id", IntegerType(), False),
    StructField("age", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("blood_group", StringType(), True),
    StructField("condition", StringType(), True),
    StructField("admission_date", StringType(), True),
    StructField("hospital", StringType(), True),
    StructField("department", StringType(), True),
    StructField("treatment_cost", DoubleType(), True),
    StructField("length_of_stay", IntegerType(), True),
    StructField("medication", StringType(), True),
    StructField("insurance", StringType(), True),
    StructField("readmitted", StringType(), True),
    StructField("city", StringType(), True)
])

df = spark.createDataFrame(data, schema)

print("Number of records:", f"{df.count():,}")
print("Number of columns:", len(df.columns))

df.show(10, truncate=False)

Number of records: 1,000,000
Number of columns: 15
+--------+----------+---+------+-----------+--------------+--------------+----------------------+----------------+--------------+--------------+------------+----------+----------+---------+
|visit_id|patient_id|age|gender|blood_group|condition     |admission_date|hospital              |department      |treatment_cost|length_of_stay|medication  |insurance |readmitted|city     |
+--------+----------+---+------+-----------+--------------+--------------+----------------------+----------------+--------------+--------------+------------+----------+----------+---------+
|1       |167622    |15 |Male  |AB+        |Asthma        |2025-04-25    |Apollo Care           |Pulmonology     |114043.22     |4             |Amlodipine  |Corporate |Yes       |Chennai  |
|2       |57315     |30 |Other |A+         |Asthma        |2025-11-29    |Sunrise Medical Center|Orthopedics     |79699.13      |8             |Metformin   |Government|No        |Mumbai   |

## 3. Inspect the Healthcare Dataset

In [10]:
df.printSchema()

df.describe(
    "age",
    "treatment_cost",
    "length_of_stay"
).show()

root
 |-- visit_id: integer (nullable = false)
 |-- patient_id: integer (nullable = false)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- blood_group: string (nullable = true)
 |-- condition: string (nullable = true)
 |-- admission_date: string (nullable = true)
 |-- hospital: string (nullable = true)
 |-- department: string (nullable = true)
 |-- treatment_cost: double (nullable = true)
 |-- length_of_stay: integer (nullable = true)
 |-- medication: string (nullable = true)
 |-- insurance: string (nullable = true)
 |-- readmitted: string (nullable = true)
 |-- city: string (nullable = true)

+-------+-----------------+-----------------+-----------------+
|summary|              age|   treatment_cost|   length_of_stay|
+-------+-----------------+-----------------+-----------------+
|  count|          1000000|          1000000|          1000000|
|   mean|        45.491256|95799.23954230067|        10.495074|
| stddev|25.96468015799698|43914.90491055637|5.7

## 4. Data Cleaning

In [11]:
# Check missing values
missing_values = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

print("Missing values:")
missing_values.show()

# Remove duplicate visit IDs
before_count = df.count()

df_clean = df.dropDuplicates(["visit_id"])

after_duplicates = df_clean.count()

print("Rows before duplicate removal:", f"{before_count:,}")
print("Rows after duplicate removal :", f"{after_duplicates:,}")

# Convert admission_date to proper date type
df_clean = df_clean.withColumn(
    "admission_date",
    to_date("admission_date", "yyyy-MM-dd")
)

# Remove invalid records
df_clean = df_clean.filter(
    (col("age") >= 0) &
    (col("age") <= 120) &
    (col("treatment_cost") > 0) &
    (col("length_of_stay") > 0) &
    col("patient_id").isNotNull() &
    col("condition").isNotNull()
)

# Create age group
df_clean = df_clean.withColumn(
    "age_group",
    when(col("age") < 18, "Child")
    .when(col("age") < 40, "Young Adult")
    .when(col("age") < 60, "Middle Aged")
    .otherwise("Senior")
)

print("Final cleaned records:", f"{df_clean.count():,}")

df_clean.show(10, truncate=False)

Missing values:
+--------+----------+---+------+-----------+---------+--------------+--------+----------+--------------+--------------+----------+---------+----------+----+
|visit_id|patient_id|age|gender|blood_group|condition|admission_date|hospital|department|treatment_cost|length_of_stay|medication|insurance|readmitted|city|
+--------+----------+---+------+-----------+---------+--------------+--------+----------+--------------+--------------+----------+---------+----------+----+
|       0|         0|  0|     0|          0|        0|             0|       0|         0|             0|             0|         0|        0|         0|   0|
+--------+----------+---+------+-----------+---------+--------------+--------+----------+--------------+--------------+----------+---------+----------+----+

Rows before duplicate removal: 1,000,000
Rows after duplicate removal : 1,000,000
Final cleaned records: 1,000,000
+--------+----------+---+------+-----------+--------------+--------------+---------

## 5. Overall Healthcare Statistics

In [13]:
from pyspark.sql.functions import countDistinct, sum, count, avg
overall_stats = df_clean.agg(
    count("visit_id").alias("total_visits"),
    countDistinct("patient_id").alias("unique_patients"),
    round(avg("age"), 2).alias("average_patient_age"),
    round(sum("treatment_cost"), 2).alias("total_treatment_cost"),
    round(avg("treatment_cost"), 2).alias("average_treatment_cost"),
    round(avg("length_of_stay"), 2).alias("average_length_of_stay")
)

overall_stats.show(truncate=False)

+------------+---------------+-------------------+--------------------+----------------------+----------------------+
|total_visits|unique_patients|average_patient_age|total_treatment_cost|average_treatment_cost|average_length_of_stay|
+------------+---------------+-------------------+--------------------+----------------------+----------------------+
|1000000     |198655         |45.49              |9.57992395423E10    |95799.24              |10.5                  |
+------------+---------------+-------------------+--------------------+----------------------+----------------------+



## 6. Patient Demographics

In [14]:
gender_analysis = df_clean.groupBy("gender").agg(
    count("patient_id").alias("patient_visits"),
    countDistinct("patient_id").alias("unique_patients")
).orderBy(desc("patient_visits"))

gender_analysis.show()

+------+--------------+---------------+
|gender|patient_visits|unique_patients|
+------+--------------+---------------+
| Other|        334036|         162348|
|Female|        333126|         162246|
|  Male|        332838|         162172|
+------+--------------+---------------+



In [15]:
age_analysis = df_clean.groupBy("age_group").agg(
    count("patient_id").alias("patient_visits"),
    countDistinct("patient_id").alias("unique_patients")
).orderBy(desc("patient_visits"))

age_analysis.show()

+-----------+--------------+---------------+
|  age_group|patient_visits|unique_patients|
+-----------+--------------+---------------+
|     Senior|        343893|         164267|
|Young Adult|        244775|         141256|
|Middle Aged|        222835|         134430|
|      Child|        188497|         122079|
+-----------+--------------+---------------+



## 7. Medical Condition Analysis

In [16]:
condition_analysis = df_clean.groupBy("condition").agg(
    count("visit_id").alias("visits"),
    countDistinct("patient_id").alias("patients"),
    round(avg("treatment_cost"), 2).alias("avg_treatment_cost"),
    round(avg("length_of_stay"), 2).alias("avg_length_of_stay"),
    round(sum("treatment_cost"), 2).alias("total_treatment_cost")
).orderBy(desc("patients"))

condition_analysis.show(truncate=False)

+--------------+------+--------+------------------+------------------+--------------------+
|condition     |visits|patients|avg_treatment_cost|avg_length_of_stay|total_treatment_cost|
+--------------+------+--------+------------------+------------------+--------------------+
|Migraine      |125003|93051   |95674.44          |10.5              |1.195959181545E10   |
|Heart Disease |125093|93015   |95831.44          |10.49             |1.198784232369E10   |
|Hypertension  |125143|92982   |95696.39          |10.48             |1.197573352299E10   |
|Diabetes      |125164|92967   |95916.75          |10.51             |1.200532455491E10   |
|Cancer        |125152|92967   |95927.23          |10.51             |1.200548472157E10   |
|Arthritis     |124907|92827   |95986.8           |10.51             |1.198942330947E10   |
|Kidney Disease|125001|92784   |95556.68          |10.5              |1.194468113103E10   |
|Asthma        |124537|92715   |95804.12          |10.48             |1.19311581

## 8. Hospital Performance

In [17]:
hospital_analysis = df_clean.groupBy("hospital").agg(
    count("visit_id").alias("visits"),
    countDistinct("patient_id").alias("patients"),
    round(avg("treatment_cost"), 2).alias("avg_treatment_cost"),
    round(avg("length_of_stay"), 2).alias("avg_length_of_stay"),
    round(sum("treatment_cost"), 2).alias("total_treatment_cost")
).orderBy(desc("total_treatment_cost"))

hospital_analysis.show(truncate=False)

+----------------------+------+--------+------------------+------------------+--------------------+
|hospital              |visits|patients|avg_treatment_cost|avg_length_of_stay|total_treatment_cost|
+----------------------+------+--------+------------------+------------------+--------------------+
|City Hospital         |200242|126460  |95876.95          |10.5              |1.919859156309E10   |
|Sunrise Medical Center|200137|126347  |95888.88          |10.48             |1.919091307993E10   |
|Metro Hospital        |200217|126470  |95732.15          |10.48             |1.916720297632E10   |
|Apollo Care           |200027|126484  |95691.04          |10.5              |1.914079166168E10   |
|Green Valley Hospital |199377|126271  |95807.14          |10.51             |1.910174026128E10   |
+----------------------+------+--------+------------------+------------------+--------------------+



## 9. Department Analysis

In [18]:
department_analysis = df_clean.groupBy("department").agg(
    count("visit_id").alias("visits"),
    round(avg("treatment_cost"), 2).alias("avg_treatment_cost"),
    round(avg("length_of_stay"), 2).alias("avg_length_of_stay"),
    round(sum("treatment_cost"), 2).alias("total_treatment_cost")
).orderBy(desc("total_treatment_cost"))

department_analysis.show(truncate=False)

+----------------+------+------------------+------------------+--------------------+
|department      |visits|avg_treatment_cost|avg_length_of_stay|total_treatment_cost|
+----------------+------+------------------+------------------+--------------------+
|Pulmonology     |143271|95855.73          |10.5              |1.373334674976E10   |
|Oncology        |142930|95960.29          |10.48             |1.371560475523E10   |
|Cardiology      |143189|95659.22          |10.5              |1.369734787583E10   |
|General Medicine|142835|95827.05          |10.5              |1.368745722747E10   |
|Neurology       |142888|95742.29          |10.49             |1.36804247215E10    |
|Nephrology      |142460|95908.12          |10.49             |1.366307022277E10   |
|Orthopedics     |142427|95641.89          |10.5              |1.362198798974E10   |
+----------------+------+------------------+------------------+--------------------+



## 10. City-wise Healthcare Analysis

In [19]:
city_analysis = df_clean.groupBy("city").agg(
    count("visit_id").alias("visits"),
    countDistinct("patient_id").alias("patients"),
    round(avg("treatment_cost"), 2).alias("avg_treatment_cost"),
    round(avg("length_of_stay"), 2).alias("avg_length_of_stay")
).orderBy(desc("visits"))

city_analysis.show(truncate=False)

+----------+------+--------+------------------+------------------+
|city      |visits|patients|avg_treatment_cost|avg_length_of_stay|
+----------+------+--------+------------------+------------------+
|Pune      |143653|102442  |95903.58          |10.51             |
|Delhi     |143313|102192  |95833.84          |10.51             |
|Mumbai    |143307|102111  |95568.85          |10.49             |
|Chennai   |142844|102037  |95929.58          |10.5              |
|Coimbatore|142748|101943  |95852.09          |10.51             |
|Hyderabad |142286|101906  |95685.51          |10.48             |
|Bangalore |141849|101731  |95821.01          |10.47             |
+----------+------+--------+------------------+------------------+



## 11. Monthly Admission Analysis

In [20]:
monthly_admissions = df_clean.withColumn(
    "month",
    date_format("admission_date", "yyyy-MM")
).groupBy("month").agg(
    count("visit_id").alias("total_admissions"),
    round(sum("treatment_cost"), 2).alias("total_treatment_cost")
).orderBy("month")

monthly_admissions.show(20, truncate=False)

+-------+----------------+--------------------+
|month  |total_admissions|total_treatment_cost|
+-------+----------------+--------------------+
|2025-01|85462           |8.19149459181E9     |
|2025-02|76845           |7.35212732813E9     |
|2025-03|84746           |8.11276536563E9     |
|2025-04|82326           |7.90312283495E9     |
|2025-05|85252           |8.17412159101E9     |
|2025-06|82118           |7.88086498337E9     |
|2025-07|84809           |8.11476591196E9     |
|2025-08|84643           |8.10072762979E9     |
|2025-09|81863           |7.83214198333E9     |
|2025-10|85200           |8.16823128734E9     |
|2025-11|82396           |7.89278013628E9     |
|2025-12|84340           |8.0760958987E9      |
+-------+----------------+--------------------+



## 12. Readmission Analysis

In [21]:
readmission_analysis = df_clean.groupBy("readmitted").agg(
    count("visit_id").alias("visits"),
    countDistinct("patient_id").alias("patients"),
    round(avg("treatment_cost"), 2).alias("avg_treatment_cost"),
    round(avg("length_of_stay"), 2).alias("avg_length_of_stay")
)

readmission_analysis.show(truncate=False)

+----------+------+--------+------------------+------------------+
|readmitted|visits|patients|avg_treatment_cost|avg_length_of_stay|
+----------+------+--------+------------------+------------------+
|No        |819000|196665  |95379.06          |10.25             |
|Yes       |181000|119096  |97700.51          |11.59             |
+----------+------+--------+------------------+------------------+



In [22]:
total_visits = df_clean.count()

readmitted_visits = df_clean.filter(
    col("readmitted") == "Yes"
).count()

readmission_rate = (readmitted_visits / total_visits) * 100

print(f"Total visits       : {total_visits:,}")
print(f"Readmitted visits  : {readmitted_visits:,}")
print(f"Readmission rate   : {readmission_rate:.2f}%")

Total visits       : 1,000,000
Readmitted visits  : 181,000
Readmission rate   : 18.10%


## 13. Readmission by Medical Condition

In [23]:
condition_readmission = df_clean.groupBy("condition").agg(
    count("visit_id").alias("total_visits"),
    sum(
        when(col("readmitted") == "Yes", 1).otherwise(0)
    ).alias("readmitted_visits")
).withColumn(
    "readmission_rate",
    round(
        col("readmitted_visits") / col("total_visits") * 100,
        2
    )
).orderBy(desc("readmission_rate"))

condition_readmission.show(truncate=False)

+--------------+------------+-----------------+----------------+
|condition     |total_visits|readmitted_visits|readmission_rate|
+--------------+------------+-----------------+----------------+
|Heart Disease |125093      |28896            |23.1            |
|Kidney Disease|125001      |28839            |23.07           |
|Cancer        |125152      |28847            |23.05           |
|Arthritis     |124907      |19041            |15.24           |
|Migraine      |125003      |19029            |15.22           |
|Asthma        |124537      |18765            |15.07           |
|Hypertension  |125143      |18812            |15.03           |
|Diabetes      |125164      |18771            |15.0            |
+--------------+------------+-----------------+----------------+



## 14. Patient Segmentation

We create patient-level features:

- Total visits
- Total treatment cost
- Average treatment cost
- Average length of stay
- Number of readmissions

Then classify patients into **Low, Medium, and High Utilization** groups.

In [24]:
patient_summary = df_clean.groupBy("patient_id").agg(
    count("visit_id").alias("total_visits"),
    round(sum("treatment_cost"), 2).alias("total_treatment_cost"),
    round(avg("treatment_cost"), 2).alias("avg_treatment_cost"),
    round(avg("length_of_stay"), 2).alias("avg_length_of_stay"),
    sum(
        when(col("readmitted") == "Yes", 1).otherwise(0)
    ).alias("readmission_count")
)

patient_summary = patient_summary.withColumn(
    "patient_segment",
    when(
        (col("total_treatment_cost") >= 200000) |
        (col("total_visits") >= 8),
        "High Utilization"
    )
    .when(
        (col("total_treatment_cost") >= 80000) |
        (col("total_visits") >= 4),
        "Medium Utilization"
    )
    .otherwise("Low Utilization")
)

patient_summary.show(20, truncate=False)

+----------+------------+--------------------+------------------+------------------+-----------------+------------------+
|patient_id|total_visits|total_treatment_cost|avg_treatment_cost|avg_length_of_stay|readmission_count|patient_segment   |
+----------+------------+--------------------+------------------+------------------+-----------------+------------------+
|29744     |11          |1152712.45          |104792.04         |10.36             |2                |High Utilization  |
|109800    |8           |674146.32           |84268.29          |11.38             |2                |High Utilization  |
|181209    |6           |619647.34           |103274.56         |9.17              |1                |High Utilization  |
|82794     |9           |1004553.9           |111617.1          |10.78             |4                |High Utilization  |
|154202    |4           |426518.99           |106629.75         |7.25              |1                |High Utilization  |
|164603    |15          

## 15. Patient Segment Summary

In [25]:
segment_summary = patient_summary.groupBy(
    "patient_segment"
).agg(
    count("patient_id").alias("patients"),
    round(avg("total_treatment_cost"), 2).alias("avg_total_cost"),
    round(avg("total_visits"), 2).alias("avg_visits"),
    round(avg("readmission_count"), 2).alias("avg_readmissions")
).orderBy(desc("patients"))

segment_summary.show(truncate=False)

+------------------+--------+--------------+----------+----------------+
|patient_segment   |patients|avg_total_cost|avg_visits|avg_readmissions|
+------------------+--------+--------------+----------+----------------+
|High Utilization  |178312  |522031.43     |5.39      |0.98            |
|Medium Utilization|17087   |148938.8      |2.05      |0.36            |
|Low Utilization   |3256    |52166.2       |1.2       |0.2             |
+------------------+--------+--------------+----------+----------------+



## 16. High-Risk / High-Utilization Patients

In [26]:
high_utilization = patient_summary.filter(
    col("patient_segment") == "High Utilization"
).orderBy(
    desc("total_treatment_cost")
)

high_utilization.show(20, truncate=False)

+----------+------------+--------------------+------------------+------------------+-----------------+----------------+
|patient_id|total_visits|total_treatment_cost|avg_treatment_cost|avg_length_of_stay|readmission_count|patient_segment |
+----------+------------+--------------------+------------------+------------------+-----------------+----------------+
|50908     |18          |1899309.14          |105517.17         |11.78             |3                |High Utilization|
|140684    |16          |1885751.3           |117859.46         |9.38              |2                |High Utilization|
|72927     |15          |1760918.95          |117394.6          |10.87             |2                |High Utilization|
|178233    |14          |1751129.49          |125080.68         |10.43             |2                |High Utilization|
|56872     |16          |1748260.76          |109266.3          |10.63             |2                |High Utilization|
|11687     |17          |1718923.53     

## 17. Top 10 Most Expensive Conditions

In [27]:
condition_analysis.orderBy(
    desc("avg_treatment_cost")
).limit(10).show(truncate=False)

+--------------+------+--------+------------------+------------------+--------------------+
|condition     |visits|patients|avg_treatment_cost|avg_length_of_stay|total_treatment_cost|
+--------------+------+--------+------------------+------------------+--------------------+
|Arthritis     |124907|92827   |95986.8           |10.51             |1.198942330947E10   |
|Cancer        |125152|92967   |95927.23          |10.51             |1.200548472157E10   |
|Diabetes      |125164|92967   |95916.75          |10.51             |1.200532455491E10   |
|Heart Disease |125093|93015   |95831.44          |10.49             |1.198784232369E10   |
|Asthma        |124537|92715   |95804.12          |10.48             |1.193115816319E10   |
|Hypertension  |125143|92982   |95696.39          |10.48             |1.197573352299E10   |
|Migraine      |125003|93051   |95674.44          |10.5              |1.195959181545E10   |
|Kidney Disease|125001|92784   |95556.68          |10.5              |1.19446811

## 18. Top 10 Hospitals by Treatment Cost

In [28]:
hospital_analysis.limit(10).show(truncate=False)

+----------------------+------+--------+------------------+------------------+--------------------+
|hospital              |visits|patients|avg_treatment_cost|avg_length_of_stay|total_treatment_cost|
+----------------------+------+--------+------------------+------------------+--------------------+
|City Hospital         |200242|126460  |95876.95          |10.5              |1.919859156309E10   |
|Sunrise Medical Center|200137|126347  |95888.88          |10.48             |1.919091307993E10   |
|Metro Hospital        |200217|126470  |95732.15          |10.48             |1.916720297632E10   |
|Apollo Care           |200027|126484  |95691.04          |10.5              |1.914079166168E10   |
|Green Valley Hospital |199377|126271  |95807.14          |10.51             |1.910174026128E10   |
+----------------------+------+--------+------------------+------------------+--------------------+



## 19. Window Function - Rank Hospitals by Treatment Cost

In [29]:
hospital_rank_window = Window.orderBy(
    desc("total_treatment_cost")
)

ranked_hospitals = hospital_analysis.withColumn(
    "revenue_rank",
    dense_rank().over(hospital_rank_window)
)

ranked_hospitals.show(truncate=False)

+----------------------+------+--------+------------------+------------------+--------------------+------------+
|hospital              |visits|patients|avg_treatment_cost|avg_length_of_stay|total_treatment_cost|revenue_rank|
+----------------------+------+--------+------------------+------------------+--------------------+------------+
|City Hospital         |200242|126460  |95876.95          |10.5              |1.919859156309E10   |1           |
|Sunrise Medical Center|200137|126347  |95888.88          |10.48             |1.919091307993E10   |2           |
|Metro Hospital        |200217|126470  |95732.15          |10.48             |1.916720297632E10   |3           |
|Apollo Care           |200027|126484  |95691.04          |10.5              |1.914079166168E10   |4           |
|Green Valley Hospital |199377|126271  |95807.14          |10.51             |1.910174026128E10   |5           |
+----------------------+------+--------+------------------+------------------+------------------

## 20. Business Insights

In [ ]:
overall = overall_stats.first()
best_condition = condition_analysis.orderBy(desc("patients")).first()
highest_cost_condition = condition_analysis.orderBy(
    desc("avg_treatment_cost")
).first()
best_hospital = hospital_analysis.orderBy(
    desc("total_treatment_cost")
).first()
highest_readmission_condition = condition_readmission.first()

print("=" * 80)
print("HEALTHCARE BUSINESS INSIGHTS")
print("=" * 80)

print(
    f"1. Total hospital visits: "
    f"{overall['total_visits']:,}"
)

print(
    f"2. Unique patients: "
    f"{overall['unique_patients']:,}"
)

print(
    f"3. Total treatment cost: "
    f"₹{overall['total_treatment_cost']:,.2f}"
)

print(
    f"4. Average treatment cost: "
    f"₹{overall['average_treatment_cost']:,.2f}"
)

print(
    f"5. Average length of stay: "
    f"{overall['average_length_of_stay']:.2f} days"
)

print(
    f"6. Most common condition: "
    f"{best_condition['condition']} "
    f"({best_condition['patients']:,} patients)"
)

print(
    f"7. Highest average treatment cost condition: "
    f"{highest_cost_condition['condition']} "
    f"(₹{highest_cost_condition['avg_treatment_cost']:,.2f})"
)

print(
    f"8. Hospital with highest treatment cost: "
    f"{best_hospital['hospital']} "
    f"(₹{best_hospital['total_treatment_cost']:,.2f})"
)

print(
    f"9. Condition with highest readmission rate: "
    f"{highest_readmission_condition['condition']} "
    f"({highest_readmission_condition['readmission_rate']:.2f}%)"
)

print(
    f"10. Overall readmission rate: "
    f"{readmission_rate:.2f}%"
)

print("=" * 80)

## 21. Visualization - Patients by Medical Condition

In [ ]:
condition_pd = condition_analysis.toPandas()

plt.figure(figsize=(11, 5))
plt.bar(
    condition_pd["condition"],
    condition_pd["patients"]
)
plt.title("Patients by Medical Condition")
plt.xlabel("Medical Condition")
plt.ylabel("Number of Patients")
plt.xticks(rotation=40)
plt.tight_layout()
plt.show()

## 22. Visualization - Treatment Cost by Condition

In [ ]:
plt.figure(figsize=(11, 5))
plt.bar(
    condition_pd["condition"],
    condition_pd["avg_treatment_cost"]
)
plt.title("Average Treatment Cost by Condition")
plt.xlabel("Medical Condition")
plt.ylabel("Average Treatment Cost (₹)")
plt.xticks(rotation=40)
plt.tight_layout()
plt.show()

## 23. Visualization - Hospital Performance

In [ ]:
hospital_pd = hospital_analysis.toPandas()

plt.figure(figsize=(11, 5))
plt.bar(
    hospital_pd["hospital"],
    hospital_pd["total_treatment_cost"]
)
plt.title("Total Treatment Cost by Hospital")
plt.xlabel("Hospital")
plt.ylabel("Treatment Cost (₹)")
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()

## 24. Visualization - Monthly Admissions

In [ ]:
monthly_pd = monthly_admissions.toPandas()

plt.figure(figsize=(13, 5))
plt.plot(
    monthly_pd["month"],
    monthly_pd["total_admissions"],
    marker="o"
)
plt.title("Monthly Hospital Admissions")
plt.xlabel("Month")
plt.ylabel("Number of Admissions")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 25. Visualization - Patient Segments

In [ ]:
segment_pd = segment_summary.toPandas()

plt.figure(figsize=(9, 5))
plt.bar(
    segment_pd["patient_segment"],
    segment_pd["patients"]
)
plt.title("Patients by Utilization Segment")
plt.xlabel("Patient Segment")
plt.ylabel("Number of Patients")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

# 🎯 Final Learning Summary

This project demonstrates an end-to-end healthcare analytics workflow using PySpark.

### PySpark concepts covered

- Creating SparkSession
- Creating DataFrames
- Defining schemas
- Data inspection
- Missing-value analysis
- Duplicate removal
- Filtering
- `withColumn()`
- `when()`
- Date functions
- `groupBy()`
- Aggregations
- `count()`
- `countDistinct()`
- `sum()`
- `avg()`
- Patient segmentation
- Readmission analysis
- Window functions
- Ranking
- Pandas conversion after aggregation
- Data visualization
- Business insights

### Business questions answered

- How many patients visited the hospitals?
- Which medical condition is most common?
- Which condition has the highest treatment cost?
- Which hospital handles the highest treatment cost?
- What is the average hospital stay?
- What is the overall readmission rate?
- Which conditions have higher readmission rates?
- Which patients are high-utilization patients?
- How do admissions change month by month?

**Important:** The dataset is completely synthetic and is intended only for demonstrating PySpark data engineering and analytics concepts.

In [ ]:
spark.stop()
print("Spark session stopped successfully.")